In [ ]:
# 03_violation_baseline.ipynb -- LightGBM baseline for the 1-4-slot-lead-time
# frequency-violation target
# !pip install lightgbm -q

import sys
sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve

import features as f

TARGET = "violation_lead"

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)
feat_df = f.build_feature_table(scada)

df = feat_df.dropna(subset=[TARGET]).copy()  # drop rows whose lead window is unresolvable

# --- Time-aware split ---
train = df[(df["date"] >= "2024-11-04") & (df["date"] <= "2025-06-30")]
val = df[(df["date"] >= "2025-07-01") & (df["date"] <= "2025-12-31")]
test = df[df["date"] >= "2026-01-01"]

print("train:", train.shape, "val:", val.shape, "test:", test.shape)
print("event rate train/val/test:", train[TARGET].mean(), val[TARGET].mean(), test[TARGET].mean())

X_train, y_train = train[f.FEATURE_COLS], train[TARGET]
X_val, y_val = val[f.FEATURE_COLS], val[TARGET]
X_test, y_test = test[f.FEATURE_COLS], test[TARGET]

# scale_pos_weight (not SMOTE) for the 2-3% positive rate -- see features.py's
# scale_pos_weight() docstring for why.
model = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, random_state=42, verbosity=-1,
    scale_pos_weight=f.scale_pos_weight(y_train),
)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)

proba_test = model.predict_proba(X_test)[:, 1]
pr_auc = average_precision_score(y_test, proba_test)
base_rate = y_test.mean()
print(f"\nPR-AUC: {pr_auc:.4f}  (random baseline = base rate = {base_rate:.4f})")

preds_05 = (proba_test >= 0.5).astype(int)
print(f"F1 @ 0.5 threshold: {f1_score(y_test, preds_05):.4f}")

# A fixed 0.5 threshold is the wrong lens for a ~2-3% positive rate -- it almost never
# fires. Report the best-F1 operating point too, which is what an actual early-warning
# system would be tuned to.
precision, recall, thresh = precision_recall_curve(y_test, proba_test)
f1s = 2 * precision * recall / (precision + recall + 1e-12)
best_idx = np.nanargmax(f1s[:-1])
print(f"Best-F1 operating point: F1={f1s[best_idx]:.4f} at threshold={thresh[best_idx]:.4f} "
      f"(precision={precision[best_idx]:.4f}, recall={recall[best_idx]:.4f})")

idx95 = np.where(precision[:-1] >= 0.95)[0]
recall_at_95p = recall[idx95].max() if len(idx95) else 0.0
print(f"Recall at >=95% precision: {recall_at_95p:.4f}")

importance = pd.Series(model.feature_importances_, index=f.FEATURE_COLS).sort_values(ascending=False)
print("\nTop 15 features:\n", importance.head(15))

# --- Results (verified 2026-07-11, LightGBM 4.6.0) ---
# PR-AUC 0.0614 vs a random/base-rate baseline of 0.0305 -- roughly 2x lift over chance,
# but low in absolute terms. F1@0.5 = 0.0000: at a naive 0.5 threshold the model almost
# never predicts positive, because the positive rate is only ~2-3%. At the best-F1
# operating point (threshold ~0.14): F1=0.1297, precision=7.4%, recall=54.6% -- i.e. a
# tuned model catches over half of upcoming violations 15-60 min ahead, at the cost of
# roughly 13 false alarms for every true one. That's a real, usable-but-noisy
# early-warning signal, not a solved problem -- reported honestly rather than dressed up
# with a single flattering metric. Recall at >=95% precision is effectively 0: a
# high-confidence-only alert mode isn't achievable yet with this feature set.
#
# Frequency violations are evidently much harder to predict with lead time than
# ramp-shocks (see 04_ramp_shock_baseline.ipynb) at this feature resolution -- plausible
# future improvements: finer time-of-day binning (violations cluster very sharply around
# 08:00-09:00 and 13:00, see 01_eda.ipynb), a shorter 1-2 slot lead horizon instead of
# 1-4, or treating this as a two-stage problem (predict ramp-shock first, then whether a
# given ramp-shock crosses into a frequency violation).


In [ ]:
# --- Appendix: shorter lead-window experiment (2026-07-11) ---
# Does shrinking the lookahead window from 1-4 slots improve the violation classifier?
# features.py's add_violation_label() and build_feature_table() both take a lead_slots
# override for exactly this test -- ramp_lead's window is untouched (that target
# already performs well, see 04_ramp_shock_baseline.ipynb).

import sys
sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import average_precision_score, precision_recall_curve

import features as f

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)
resid = f.build_study1_residual_signal()  # computed once, reused across all window sizes


def run(lead_slots):
    feat = f.build_feature_table(scada, study1_residual=resid, violation_lead_slots=lead_slots)
    df = feat.dropna(subset=["violation_lead"]).copy()
    train = df[(df["date"] >= "2024-11-04") & (df["date"] <= "2025-06-30")]
    val = df[(df["date"] >= "2025-07-01") & (df["date"] <= "2025-12-31")]
    test = df[df["date"] >= "2026-01-01"]

    X_train, y_train = train[f.FEATURE_COLS], train["violation_lead"]
    X_val, y_val = val[f.FEATURE_COLS], val["violation_lead"]
    X_test, y_test = test[f.FEATURE_COLS], test["violation_lead"]

    model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbosity=-1,
                                scale_pos_weight=f.scale_pos_weight(y_train))
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(30, verbose=False)])

    proba = model.predict_proba(X_test)[:, 1]
    pr_auc = average_precision_score(y_test, proba)
    base_rate = y_test.mean()

    precision, recall, thresh = precision_recall_curve(y_test, proba)
    f1s = 2 * precision * recall / (precision + recall + 1e-12)
    best_idx = np.nanargmax(f1s[:-1])

    print(f"lead_slots={lead_slots}: base_rate={base_rate:.4f}  PR-AUC={pr_auc:.4f} "
          f"(lift={pr_auc / base_rate:.2f}x)  best-F1={f1s[best_idx]:.4f} "
          f"(P={precision[best_idx]:.4f} R={recall[best_idx]:.4f})")


for k in [4, 3, 2, 1]:
    run(k)

# --- Findings (verified 2026-07-11) ---
# lead_slots=4 (current, 15-60 min): base_rate=3.05%  PR-AUC=0.0614 (2.01x lift)  best-F1=0.1297 (P=7.4%  R=54.6%)
# lead_slots=3 (15-45 min):          base_rate=2.46%  PR-AUC=0.0487 (1.98x lift)  best-F1=0.1119 (P=6.2%  R=58.6%)
# lead_slots=2 (15-30 min):          base_rate=1.81%  PR-AUC=0.0562 (3.10x lift)  best-F1=0.1486 (P=9.0%  R=41.9%)
# lead_slots=1 (15 min only):        base_rate=1.11%  PR-AUC=0.0828 (7.47x lift)  best-F1=0.1481 (P=10.7% R=24.0%)
#
# Not a clean "shorter is better" result -- it's a genuine trade-off, not a win, and one
# data point (3 slots) is even worse than the current default on every metric, which
# argues against reading too much into small window changes. What IS consistent: lift
# over the (shrinking) base rate rises sharply as the window narrows -- 7.47x at 1 slot
# vs 2.01x at 4 -- because near-term prediction is fundamentally easier. But recall pays
# for it: a 1-slot (15-min-only) model catches under a quarter of violations, vs over
# half at 4 slots. 2 slots looks like the best balance (best F1 of all four, meaningfully
# better lift than the current default, and still ~42% recall) but "best" here depends on
# what the eventual dashboard/alert consumer actually wants -- more warnings caught with
# more noise (4 slots) vs fewer, higher-confidence warnings with less lead time (1-2
# slots). Left as an open product decision, not resolved by picking a "winner" here --
# the shipped default in 02_features.ipynb and predict.py remains 4 slots (matching the
# roadmap's original 1-4-slot scope) unless that product call is made explicitly.
